# 01 — Kolmogorov Flow: Data Exploration

Downloads `KolmFlow_valid_256.h5` (~5 GB) if missing, computes normalization statistics, and visualizes sample vorticity fields.

Runs in local Jupyter or Colab. In Colab, Google Drive is mounted automatically by `src.env`.

In [ ]:
# --- Bootstrap: locate repo, make `src` importable ---
import sys, os
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    # EDIT this path if your Drive layout differs
    REPO = Path('/content/drive/MyDrive/courses/24788-Intro_of_DL/project/code')
except ImportError:
    IN_COLAB = False
    REPO = Path.cwd()
    while not (REPO / 'requirements.txt').exists() and REPO != REPO.parent:
        REPO = REPO.parent

assert (REPO / 'src' / 'env.py').exists(), f'repo not found at {REPO}'
sys.path.insert(0, str(REPO))
print(f'IN_COLAB = {IN_COLAB}\nREPO     = {REPO}')

In [ ]:
from src.env import DATA_DIR, RESULTS_DIR, summary
from src import data as D
print(summary())

In [ ]:
# --- Download (5 GB, ~5-10 min on Colab) ---
path = D.download_if_needed()
print(f'Data: {path} ({path.stat().st_size / 1e9:.2f} GB)')

In [ ]:
# --- Inspect H5 structure ---
import h5py
with h5py.File(path, 'r') as f:
    def show(name, obj):
        shape = getattr(obj, 'shape', None)
        dtype = getattr(obj, 'dtype', None)
        print(f'  {name:30s}  shape={shape}  dtype={dtype}')
    f.visititems(show)
    u = f['valid']['u']
    print(f'\nu stats on trajectory 0: min={u[0].min():.3f} max={u[0].max():.3f} mean={u[0].mean():.3f}')

In [ ]:
# --- Train/val/test split (by trajectory, to prevent leakage) ---
train_ids, val_ids, test_ids = D.get_trajectory_split()
print(f'Train trajectories: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}')
print(f'  train frames: {len(train_ids) * D.N_TIMESTEPS}')
print(f'  val   frames: {len(val_ids)   * D.N_TIMESTEPS}')
print(f'  test  frames: {len(test_ids)  * D.N_TIMESTEPS}')

In [ ]:
# --- Compute normalization stats from a subset of training trajectories ---
import json
stats = D.compute_norm_stats(path, train_ids=train_ids, n_sample_traj=30)
print(f'mean = {stats.mean:.6f}')
print(f'std  = {stats.std:.6f}')
with open(DATA_DIR / 'norm_stats.json', 'w') as f:
    json.dump(stats.to_dict(), f, indent=2)
print(f'Saved → {DATA_DIR / "norm_stats.json"}')

In [ ]:
# --- Visualize: one trajectory, evenly-spaced timesteps ---
import matplotlib.pyplot as plt
import numpy as np

ds = D.KolmFlowFrames(split='train', norm=None, burn_in=0)
traj = ds.traj_ids[0]
with h5py.File(path, 'r') as f:
    traj_data = np.asarray(f['valid']['u'][int(traj)])  # [200, 160, 160]

ts = np.linspace(0, D.N_TIMESTEPS - 1, 8, dtype=int)
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
vmin, vmax = traj_data.min(), traj_data.max()
for ax, t in zip(axes, ts):
    ax.imshow(traj_data[t], cmap='RdBu_r', vmin=vmin, vmax=vmax)
    ax.set_title(f't={t}')
    ax.axis('off')
fig.suptitle(f'Trajectory {traj}: vorticity evolution')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'trajectory_evolution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualize: diversity across trajectories (single timestep) ---
t_show = 150  # late enough to be fully turbulent
with h5py.File(path, 'r') as f:
    samples = np.stack([np.asarray(f['valid']['u'][int(i), t_show]) for i in train_ids[:8]])

fig, axes = plt.subplots(1, 8, figsize=(20, 3))
vmin, vmax = samples.min(), samples.max()
for ax, s, i in zip(axes, samples, train_ids[:8]):
    ax.imshow(s, cmap='RdBu_r', vmin=vmin, vmax=vmax)
    ax.set_title(f'traj {int(i)}')
    ax.axis('off')
fig.suptitle(f'8 different trajectories at t={t_show}')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'trajectory_diversity.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Vorticity value distribution ---
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(traj_data.reshape(-1), bins=80, density=True, alpha=0.7)
ax.axvline(stats.mean, color='k', linestyle='--', label=f'mean={stats.mean:.2f}')
ax.axvline(stats.mean + stats.std, color='r', linestyle=':', label=f'+1 std')
ax.axvline(stats.mean - stats.std, color='r', linestyle=':')
ax.set_xlabel('vorticity'); ax.set_ylabel('density'); ax.legend()
ax.set_title('Vorticity distribution (1 trajectory)')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'vorticity_histogram.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Sanity-check the Dataset class: one sample round-trip ---
ds = D.KolmFlowFrames(split='train', norm=stats)
x = ds[0]
print(f'Sample shape: {tuple(x.shape)}  dtype: {x.dtype}')
print(f'After normalization: mean={x.mean():.4f}  std={x.std():.4f}')
print(f'Dataset length (train): {len(ds)}')